# Session 5 - Reconstruct, Verify, Ship
## AI Productization and Final Demo

> **A demo is not evidence until it survives a restart.**

The same evidence chain built earlier is carried into a runnable application:

**DATA CONTRACT -> LEGITIMATE MODEL -> EXPLANATION -> RELIABILITY ROUTER -> ARTIFACT BUNDLE -> APP -> DEPLOYMENT**

### Final target
A reproducible **Reliable AI Risk Intelligence** application will be rebuilt from source and persisted artifacts, verified in a new process, exercised through all four Session 4 routes, and made ready for local or internet deployment.

### Fixed boundary
- No new model shopping is performed.
- No new threshold is tuned from 2025.
- The 2025 holdout remains final evaluation evidence.
- Productization changes packaging and verification, not the scientific decision rule.


## 0. Course continuity - what is being carried into Session 5?

| Earlier work | Evidence retained | Session 5 consequence |
|---|---|---|
| Session 1 - data fusion | 840-row modeling contract | the application uses the same engineered schema |
| Session 2 - leakage-aware selection | `logistic` is the best eligible path; the leaky candidate is ineligible | the selected object is persisted, not a favorite model name |
| Session 3 - explainability/debugging | global reliance + local evidence + error-case reasoning | accepted decisions can carry explanation evidence |
| Session 4 - reliable AI | calibration + contract + anomaly + confidence routing | the four route states become the application contract |
| **Session 5 - productization** | source + artifacts + tests + app | the behavior must survive restart and deployment |

**Reference checkpoints:** 840 rows, 16 model features, `logistic`, 2025 Macro F1 about 0.6357, temperature about 1.0883, review threshold 0.650, anomaly threshold about -0.0404.


## Live build format

Seven short engineering challenges are used. Every challenge follows the same pattern:

**PREDICT -> BUILD/BREAK -> RUN -> INSPECT -> DEFEND**

| Round | Challenge |
|---|---|
| 0 | What disappears after a restart? |
| 1 | Which files are actually part of the product? |
| 2 | Can metadata corruption be detected? |
| 3 | Can all four Session 4 routes be reproduced after loading? |
| 4 | What must a human-review record contain? |
| 5 | Which regression test catches which failure? |
| 6 | Can a clean process verify the full product? |
| 7 | Which deployment path fits the goal: local, temporary public, or hosted? |

A wrong prediction is useful when the reasoning behind it can be defended and then corrected by evidence.


## 1. Find the course package and import the shared runtime

The notebook is not allowed to become a second implementation. The `src/` package remains the source of truth.


In [ ]:
import json
import shutil
import subprocess
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    if (candidate / "src" / "reliable_ai").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("A project root containing src/reliable_ai could not be found.")

SRC_DIR = ROOT / "src"
DATA_DIR = SRC_DIR / "data"
MODEL_DIR = SRC_DIR / "model"
REPORTS_DIR = SRC_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from reliable_ai import (
    CLASS_NAMES,
    LEGITIMATE_MODEL_NAMES,
    MODEL_FEATURES,
    POLICY_VERSION,
    REQUIRED_ARTIFACT_FILES,
    encode_target,
    fit_and_persist_artifacts,
    load_artifacts,
    run_training_comparison,
    validate_artifact_contract,
)
from app import get_app, predict_request
from verify_product import DEMO_CASES, EXPECTED

print("Project root:", ROOT)
print("Source path :", SRC_DIR)
print("Class order :", list(CLASS_NAMES))
print("Features    :", len(MODEL_FEATURES))


### ROUND 0 - What disappears after a restart?

A notebook has just produced a correct decision. The kernel is restarted.

Which item is most likely to disappear while the files remain intact?

**A.** `metadata.json`  
**B.** an already-fitted Python object stored only in memory  
**C.** `risk_model.joblib`  
**D.** `threshold_analysis.csv`

Lock one choice and one reason before the answer is revealed.


In [ ]:
round0_choice = "TODO"  # A / B / C / D
round0_reason = "TODO: one precise sentence"
print("Locked:", round0_choice, "|", round0_reason)


## 2. Rebuild the evidence-selected path from the established data contract

The candidate registry is rerun only to reconstruct the already-defined training path. Eligibility remains the first gate.


In [ ]:
comparison = run_training_comparison(DATA_DIR)
metrics = comparison["metrics"].copy()
metrics["eligible"] = metrics["model"].isin(LEGITIMATE_MODEL_NAMES)

selected_name = comparison["selected_model_name"]
selected_model = comparison["selected_model"]
selected_features = list(comparison["selected_features"])
split = comparison["split"]
train, calibration, test = split["train"], split["calibration"], split["test"]

view = metrics.sort_values(["eligible", "macro_f1", "log_loss"], ascending=[False, False, True]).reset_index(drop=True)
display(view)
print("Selected eligible model:", selected_name)
print(f"Split: train={len(train)} | calibration={len(calibration)} | test={len(test)}")

assert len(train) == 600 and len(calibration) == 120 and len(test) == 120
assert selected_name == "logistic"
assert selected_features == list(MODEL_FEATURES)
assert metrics.loc[metrics["model"] == "advanced_leaky", "macro_f1"].iloc[0] > metrics.loc[metrics["model"] == selected_name, "macro_f1"].iloc[0]


### Evidence reading

The high leaky score is deliberately visible. It is rejected because the feature contract makes it ineligible, not because its score is small. This is the direct bridge from Session 2 into productization.


## 3. Rebuild and persist the Session 4 reliability contract

The shared `src` implementation is now used to persist the **selected model + calibration + anomaly model + reference medians + policy evidence + metadata + SHA-256 manifest** as one coherent bundle.


In [ ]:
persisted = fit_and_persist_artifacts(comparison, DATA_DIR, MODEL_DIR)
artifacts = load_artifacts(MODEL_DIR, verify=True)
meta = artifacts["metadata"]

checkpoint = pd.DataFrame([
    ["selected model", meta["selected_model_name"]],
    ["2025 Macro F1", meta["selected_test_metrics"]["macro_f1"]],
    ["temperature", meta["temperature"]],
    ["review threshold", meta["review_threshold"]],
    ["anomaly threshold", meta["anomaly_threshold"]],
    ["policy version", meta["policy_version"]],
], columns=["checkpoint", "value"])
display(checkpoint)

assert meta["selected_model_name"] == "logistic"
assert abs(meta["selected_test_metrics"]["macro_f1"] - 0.6357) < 1e-4
assert abs(meta["temperature"] - 1.0883) < 0.02
assert abs(meta["review_threshold"] - 0.650) < 1e-12
assert abs(meta["anomaly_threshold"] - (-0.040398)) < 0.01
assert meta["policy_version"] == POLICY_VERSION


## 4. Report the frozen policy on 2025 - do not tune it

The final holdout is used only to report behavior under the already-selected model and already-selected policy.


In [ ]:
X_test = test[MODEL_FEATURES]
y_test = encode_target(test["risk_label"])
raw = artifacts["model"].predict_proba(X_test)
calibrated = artifacts["temperature_scaler"].transform(raw)
pred = calibrated.argmax(axis=1)
conf = calibrated.max(axis=1)
accepted = conf >= artifacts["review_threshold"]

from sklearn.metrics import f1_score
macro_f1 = float(f1_score(y_test, pred, average="macro"))
coverage = float(accepted.mean())
review_rate = float(1 - coverage)
selective_accuracy = float((pred[accepted] == y_test[accepted]).mean())

display(pd.DataFrame({
    "metric": ["macro_f1", "coverage", "review_rate", "selective_accuracy"],
    "2025": [macro_f1, coverage, review_rate, selective_accuracy],
}))
print("The test year is being measured here, not optimized.")


### ROUND 1 - Artifact detective

Classify the files below into three groups:

- **runtime/reconstruction** - required to reproduce behavior;
- **decision evidence** - not called for every request, but needed to defend the product;
- **incidental** - useful perhaps, but not part of this product contract.

Candidates: `risk_model.joblib`, `temperature_scaler.joblib`, `anomaly_model.joblib`, `reference_medians.joblib`, `metadata.json`, `model_comparison.csv`, `threshold_analysis.csv`, `screenshot.png`, `notes.txt`.


In [ ]:
runtime = ["TODO"]
evidence = ["TODO"]
incidental = ["TODO"]
print("RUNTIME:", runtime)
print("EVIDENCE:", evidence)
print("INCIDENTAL:", incidental)


## 5. Inspect the persisted artifact contract

The manifest is a byte-level identity check. It answers **which files were deployed**, not **whether the scientific model is good**.


In [ ]:
contract = validate_artifact_contract(MODEL_DIR)
assert contract["ok"], contract["problems"]
manifest_rows = []
for name, info in contract["manifest"]["files"].items():
    manifest_rows.append({"artifact": name, "bytes": info["bytes"], "sha256_prefix": info["sha256"][:16]})
display(pd.DataFrame(manifest_rows))
print("Artifact contract: PASSED")


### ROUND 2 - Contract sabotage

A copy of `metadata.json` will be changed while the model bytes remain untouched.

Predict the result before execution:

1. Can Python still read the JSON?
2. Is it still the same declared product?
3. Which check should fail first: feature-order identity or model accuracy?


In [ ]:
python_can_read = "TODO"
same_product = "TODO"
first_failure = "TODO"
print({"python_can_read": python_can_read, "same_product": same_product, "first_failure": first_failure})


## 6. Break a copied bundle safely, then prove that the validator catches it

The real `src/model/` directory is not modified. A temporary copy is sabotaged.


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    broken_dir = Path(tmp) / "model"
    shutil.copytree(MODEL_DIR, broken_dir)
    meta_path = broken_dir / "metadata.json"
    broken_meta = json.loads(meta_path.read_text(encoding="utf-8"))
    broken_meta["model_features"] = list(reversed(broken_meta["model_features"]))
    meta_path.write_text(json.dumps(broken_meta, indent=2), encoding="utf-8")

    broken = validate_artifact_contract(broken_dir)
    print("Contract OK:", broken["ok"])
    print("Detected problems:")
    for issue in broken["problems"]:
        print(" -", issue)
    assert not broken["ok"]
    assert any("model_features" in issue or "fingerprint mismatch" in issue for issue in broken["problems"])

print("The original bundle remains valid:", validate_artifact_contract(MODEL_DIR)["ok"])


### ROUND 3 - Route arena

Four requests from Session 4 are loaded below. Predict the route before the router is executed:

- `ACCEPT_CASE`
- `LOW_CONFIDENCE_CASE`
- `ANOMALY_CASE`
- `INVALID_CASE`

Allowed states: `ACCEPT_AI_DECISION`, `HUMAN_REVIEW_LOW_CONFIDENCE`, `HUMAN_REVIEW_ANOMALY`, `REJECT_INVALID_INPUT`.


In [ ]:
route_prediction = {
    "ACCEPT_CASE": "TODO",
    "LOW_CONFIDENCE_CASE": "TODO",
    "ANOMALY_CASE": "TODO",
    "INVALID_CASE": "TODO",
}
route_prediction


## 7. Reproduce all four Session 4 routes from loaded artifacts

The UI is bypassed here. The shared application boundary is called directly.


In [ ]:
artifacts = load_artifacts(MODEL_DIR, verify=True)
route_results = {}
rows = []
for name, request in DEMO_CASES.items():
    result = predict_request(request, artifacts)
    route_results[name] = result
    assert result["status"] == EXPECTED[name], (name, result)
    rows.append({
        "case": name,
        "status": result["status"],
        "reason": result["reason"],
        "prediction": result["prediction"],
        "confidence": result["confidence"],
        "anomaly_score": result["anomaly_score"],
        "problems": len(result["problems"]),
    })
display(pd.DataFrame(rows))
print("All four routes were reproduced from the persisted bundle.")


### Route semantics

- `REJECT_INVALID_INPUT` = hard input contract failure; no prediction is produced.
- `HUMAN_REVIEW_ANOMALY` = valid but unfamiliar input.
- `HUMAN_REVIEW_LOW_CONFIDENCE` = valid and familiar, but below the automatic-action confidence rule.
- `ACCEPT_AI_DECISION` = every earlier gate passed; acceptance is not certainty.


### ROUND 4 - Human-review ticket

A reviewer should not need to reopen the notebook to understand why a case was deferred.

Choose the minimum fields that must travel with a review case. Include enough information to identify the route, thresholds, model/policy version, and useful explanation evidence.


In [ ]:
review_fields = ["TODO"]
print("Proposed fields:", review_fields)


## 8. Export and verify one human-review JSON record

Only `HUMAN_REVIEW_*` outcomes are accepted by this helper.


In [ ]:
def export_review_record(result: dict, path: Path):
    # LIVE BUILD: replace this starter body with the review-record contract.
    # Required work: HUMAN_REVIEW_* guard -> compact evidence dict -> JSON write/read verification.
    print("TODO checkpoint: implement the review-record contract here.")
    return None

# This starter cell is intentionally safe to execute before the function is completed.


## 9. Keep the interface thin

The application is allowed to collect inputs and render evidence. It is **not** allowed to engineer a second feature path, fit calibration, choose thresholds, or recreate routing logic.

`UI INPUT -> predict_request() -> make_safe_decision() -> structured evidence -> UI`


In [ ]:
app_path = SRC_DIR / "app.py"
app_text = app_path.read_text(encoding="utf-8")
checks = {
    "shared decision function used": "make_safe_decision" in app_text,
    "deployment flag available": "--share" in app_text,
    "artifact verification used": "load_artifacts" in app_text,
}
print(checks)
assert all(checks.values())

demo = get_app()
print("Gradio app object:", type(demo).__name__)
assert demo is not None


### ROUND 5 - Regression-test detective

Match each failure to the strongest regression check:

**A.** feature order in metadata was reversed  
**B.** calibration data leaked into the test year  
**C.** `advanced_leaky` became selectable  
**D.** calibrated probabilities no longer sum to one  
**E.** a model file was replaced without updating the manifest

Checks:
1. chronological split invariant  
2. prohibited-candidate selection test  
3. probability-distribution invariant  
4. artifact contract / fingerprint test  
5. metadata feature-schema test


In [ ]:
test_match = {"A": "TODO", "B": "TODO", "C": "TODO", "D": "TODO", "E": "TODO"}
print(test_match)


## 10. Execute the regression contract

The suite includes the earlier data/model/reliability checks plus Session 5 manifest tampering and audit-field checks.


In [ ]:
pytest_proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(SRC_DIR / "tests" / "test_system.py")],
    capture_output=True,
    text=True,
    cwd=str(SRC_DIR),
)
print(pytest_proc.stdout)
if pytest_proc.stderr:
    print(pytest_proc.stderr)
assert pytest_proc.returncode == 0


### ROUND 6 - Final boss: cold-start verification

Before execution, name one hidden dependency that a notebook `Run All` could accidentally provide but a new process cannot.


In [ ]:
hidden_dependency = "TODO"
why_new_process_is_stronger = "TODO"
print(hidden_dependency, "|", why_new_process_is_stronger)


## 11. Run the cold-start product verifier

This command starts a separate Python process, validates fingerprints and metadata, loads the product, and reproduces all four routes.


In [ ]:
verify_proc = subprocess.run(
    [sys.executable, str(SRC_DIR / "verify_product.py")],
    capture_output=True,
    text=True,
    cwd=str(SRC_DIR),
)
print(verify_proc.stdout)
if verify_proc.stderr:
    print(verify_proc.stderr)
assert verify_proc.returncode == 0
assert "PRODUCT VERIFICATION PASSED" in verify_proc.stdout


### ROUND 7 - Ship it: choose the deployment path

Three deployment goals are available:

**LOCAL** - fastest and most private; runs only on the current machine/network configuration.  
**TEMPORARY PUBLIC** - `python app.py --share`; useful for a short live demonstration.  
**HOSTED GRADIO SPACE** - upload the `src/` runtime to a Hugging Face Gradio Space for a persistent internet demo under the platform's available free CPU quota.

Choose one path for each scenario:

1. A five-minute classroom link with no cloud setup.
2. A persistent portfolio/demo URL.
3. A disconnected or privacy-sensitive demonstration.


In [ ]:
deployment_choices = {1: "TODO", 2: "TODO", 3: "TODO"}
print(deployment_choices)


## 12. Deployment commands - one codebase, three launch modes

### Local
```bash
cd src
python -m pip install -r requirements.txt
python train_model.py
python verify_product.py
python app.py
```

### Temporary public Gradio link
```bash
cd src
python app.py --share
```
The link depends on Gradio's sharing service and is intended for short demonstrations.

### Hugging Face Spaces
Create a **Gradio** Space and upload the **contents of `src/`** so that `app.py`, `requirements.txt`, `reliable_ai/`, `model/`, and `data/` are at the Space repository root. The app can then be rebuilt by the hosted environment. A free CPU tier may be used when it is available within the account/platform quota.

The detailed checklist is stored in `src/DEPLOYMENT.md`.


In [ ]:
required_deploy_files = [
    SRC_DIR / "app.py",
    SRC_DIR / "requirements.txt",
    SRC_DIR / "DEPLOYMENT.md",
    SRC_DIR / "verify_product.py",
    MODEL_DIR / "artifact_manifest.json",
]
for path in required_deploy_files:
    print("OK" if path.exists() else "MISSING", "-", path.relative_to(ROOT))
assert all(path.exists() for path in required_deploy_files)

smoke_proc = subprocess.run(
    [sys.executable, str(SRC_DIR / "app.py"), "--smoke-only", "--no-browser"],
    capture_output=True,
    text=True,
    cwd=str(SRC_DIR),
)
print(smoke_proc.stdout)
assert smoke_proc.returncode == 0


## 13. Final evidence record

The application is considered ready for demonstration only when the data/model/policy/software chain is visible in one record.


In [ ]:
final_record = {
    "data_rows": 840,
    "model_features": len(MODEL_FEATURES),
    "selected_model": meta["selected_model_name"],
    "future_2025_macro_f1": round(macro_f1, 4),
    "temperature": round(meta["temperature"], 4),
    "review_threshold": round(meta["review_threshold"], 3),
    "anomaly_threshold": round(meta["anomaly_threshold"], 6),
    "coverage_2025": round(coverage, 4),
    "review_rate_2025": round(review_rate, 4),
    "selective_accuracy_2025": round(selective_accuracy, 4),
    "artifact_contract": "PASSED" if validate_artifact_contract(MODEL_DIR)["ok"] else "FAILED",
    "four_routes": "PASSED",
    "cold_start": "PASSED" if verify_proc.returncode == 0 else "FAILED",
    "regression_tests": "PASSED" if pytest_proc.returncode == 0 else "FAILED",
    "deployment_smoke": "PASSED" if smoke_proc.returncode == 0 else "FAILED",
}
final_path = REPORTS_DIR / "session5_final_evidence.json"
final_path.write_text(json.dumps(final_record, indent=2), encoding="utf-8")
print(json.dumps(final_record, indent=2))
print("Saved:", final_path)


## Final demonstration sequence

1. **DATA:** show the 840-row contract and 16 approved model features.
2. **SELECT:** show `advanced_leaky` scoring higher but remaining ineligible; persist `logistic`.
3. **INSPECT:** connect to one Session 3 explanation and its limitation.
4. **CONTROL:** reproduce ACCEPT / LOW-CONFIDENCE REVIEW / ANOMALY REVIEW / REJECT.
5. **RECONSTRUCT:** validate metadata + fingerprints in a clean process.
6. **VERIFY:** run the regression suite and smoke check.
7. **SHIP:** open the local app, request a temporary public link, or publish the same runtime to a Gradio Space.

> The interface is demonstrated after the evidence chain, not instead of it.


## Technical defense and release checks

A release can be defended when the following questions have concrete answers:

- Why is the leaky 0.96-class score invalid?
- Why must feature order travel with the model?
- Why are `T`, the review threshold, and the anomaly threshold product identity?
- Why is anomaly review different from invalid-input rejection?
- What does a SHA-256 fingerprint prove, and what does it not prove?
- Which test detects stale or tampered artifacts?
- Why is one shared decision function used by the notebook, app, and tests?
- Which risks remain outside the regression suite: future drift, bias, causality, and untested security boundaries?

### Final result
**The controlled AI behavior is portable, testable, auditable, and deployable from the same source package.**


## Assignment 05 - Final AI Product and Defense

Required evidence:

1. runnable package with `requirements.txt` and exact launch commands;
2. persisted model/reliability bundle with a valid `artifact_manifest.json`;
3. all four route states demonstrated;
4. one verified human-review JSON record;
5. at least three additional edge/regression tests, each linked to a specific failure;
6. local or hosted demo path documented;
7. cold-start verification output;
8. concise limitations and AI-assistance disclosure.

A polished interface without reconstructable decision evidence is incomplete.
